# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [12]:
# Initialize and constants

load_dotenv(override=True)
ollama_base_url = os.getenv('OLLAMA_BASE_URL')
MODEL = os.getenv('OLLAMA_MODEL')

openai = OpenAI(base_url=ollama_base_url, api_key="anything")

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://wordpress.com']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://wordpress.com


In [14]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers/jobs page', 'url': 'https://edwarddonner.com/careers/'},
  {'type': 'contact us page', 'url': 'https://edwarddonner.com/contact/'}]}

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [17]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com'},
  {'type': 'about page', 'url': 'https://www.edwarddonner.com'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/blog'},
  {'type': 'team bio page', 'url': 'https://www.edwarddonner.com/team/'},
  {'type': 'shop/cart page', 'url': 'https://store.edwarddonner.com'}]}

In [21]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'blog', 'url': 'https://blog.huggingface.co/'},
  {'type': 'docs hub', 'url': 'https://huggingface.co/docs'},
  {'type': 'github repository', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter handle', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord server', 'url': 'https://join.discord.org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [37]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    all_links = relevant_links.get('links', [])
    for key, val in relevant_links.items():
        if key != 'links' and isinstance(val, list):
            all_links += val

    for link in all_links:
        # Xử lý cả trường hợp link là string hoặc dict
        if isinstance(link, str):
            link_url = link
            link_type = "page"
        else:
            link_url = link.get("url", "")
            link_type = link.get("type", "page")

        if not link_url or not link_url.startswith("http"):
            continue
        try:
            result += f"\n\n### Link: {link_type}\n"
            result += fetch_website_contents(link_url)
        except Exception as e:
            print(f"Skipping {link_url}: {e}")
            continue
    return result


In [26]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
openbmb/MiniCPM5-1B
Updated
3 days ago
•
15.6k
•
510
meituan-longcat/LongCat-Video-Avatar-1.5
Updated
3 days ago
•
373
bytedance-research/Lance
Updated
1 day ago
•
2.51k
•
959
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
Apr 17
•
1.96M
•
1.02k
nvidia/LocateAnything-3B
Updated
2

In [27]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [28]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [29]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links
Skipping https://blog.huggingface.co: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10fd97530>: Failed to resolve 'blog.huggingface.co' ([Errno 8] nodename nor servname provided, or not known)"))
Skipping https://docs.huggingface.co: HTTPSConnectionPool(host='docs.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x11819cec0>: Failed to resolve 'docs.huggingface.co' ([Errno 8] nodename nor servname provided, or not known)"))


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nopenbmb/MiniCPM5-1B\nUpdated\n3 days ago\n•\n15.6k\n•\n510\nmeituan-longcat/LongCat-Video-Avatar-1.5\nUpdated\n3 days ago\n•\n

In [32]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [33]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links
Skipping https://blog.huggingface.co/: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x1182320c0>: Failed to resolve 'blog.huggingface.co' ([Errno 8] nodename nor servname provided, or not known)"))
Skipping https://docs.huggingface.co/: HTTPSConnectionPool(host='docs.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x118280ec0>: Failed to resolve 'docs.huggingface.co' ([Errno 8] nodename nor servname provided, or not known)"))
Skipping https://join.discord.com/huggingface: HTTPSConnectionPool(host='join.discord.com', port=443): Max retries exceeded with url: /huggingface (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10e65fbc0>: Failed to resolve 'j

# Hugging Face: Building a Future in AI Collaboration

At Hugging Face, we're passionate about empowering the machine learning community to build, discover, and collaborate on better models. Our platform is designed to bridge the gap between research and industry applications.

## About Us

Hugging Face is an open-source collaboration platform for machine learning (ML) researchers and practitioners. We believe that collaboration is key to accelerating AI advancements. By hosting millions of pre-trained AI models, datasets, and applications, we provide a one-stop-shop for researchers, developers, and businesses to explore the vast potential of AI.

## Our Culture

We foster a community-driven culture where individuals can share knowledge, resources, and expertise to push the boundaries of AI research. Our platform promotes transparency, reproducibility, and accessibility, ensuring that everyone can benefit from cutting-edge AI technologies.

## Products and Services

* [Hugging Face Models](https://huggingface.co/models): Browse over 2 million pre-trained models, including audio-driven talking-head video generation and high-fidelity image-to-3D generation.
* [Hugging Face Datasets](https://huggingface.co/datasets): Explore 500,000+ datasets covering various domains, from text to vision and speech recognition.
* [HuggyChat](https://huggingchat.com/): Engage with the Hugging Face community through our chatbot platform.

## Solutions for

* **Businesses**: Accelerate AI adoption with our enterprise-ready solutions, including Inference Providers, Storage Buckets, and more.
* **Researchers**: Develop, test, and collaborate on cutting-edge ML models using our free and open-source platform.
* **Developers**: Integrate popular AI frameworks like Transformers into your applications.

## Career Opportunities

Join us in shaping the future of AI! Hugging Face is an equal opportunity employer offering competitive salaries, benefits, and a dynamic work environment. Explore our [careers page](https://huggingface.com/careers) for current job openings and our mission to empower the ML community.

## Get Started Today!

Take your machine learning journey to the next level by exploring Hugging Face's vast collection of models, datasets, and applications. Join our vibrant community today and discover how we can work together to build a brighter future in AI.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [35]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [38]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links
Skipping https://blog.huggingface.co/: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10fa8eba0>: Failed to resolve 'blog.huggingface.co' ([Errno 8] nodename nor servname provided, or not known)"))
Skipping https://join.discord.com/huggingface: HTTPSConnectionPool(host='join.discord.com', port=443): Max retries exceeded with url: /huggingface (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10f807e00>: Failed to resolve 'join.discord.com' ([Errno 8] nodename nor servname provided, or not known)"))


# Hugging Face: Empowering the Future of Artificial Intelligence

At Hugging Face, we're building a community that's passionate about making progress in artificial intelligence. Our platform is designed to facilitate collaboration among machine learning practitioners, researchers, and developers.

## What We Do

We create the tools and platforms needed to advance AI research, development, and application. Our flagship offering is a collaborative platform where the global machine learning community can share and work with models, datasets, and applications.

## Models & Datasets

With over 2 million pre-trained models, our platform provides an unparalleled opportunity for users to discover, explore, and deploy AI-based solutions. Our dataset collection spans various niches, including but not limited to:

*   Wikidata
*   Structured Wikipedia
*   Open WebText
*   Ultra-Fine Web

From audio-driven talking-head video generation to high-fidelity image-to-3D generation, our portfolio of models showcases cutting-edge AI innovations.

## Spaces & Applications

Our platform features a dedicated space for hosting and collaborating on models, where users can work together to advance the state-of-the-art in machine learning. With applications spanning video production, text editing, data augmentation as well as image-to-3D generation, we're fostering diverse and innovative use cases across industries.

## Hugging Face PRO & Team

We offer a subscription-based model tailored for businesses looking to deploy and manage AI solutions at scale.

Whether an aspiring researcher or seasoned developer working in industries like education, automotive, retail or entertainment, our user-centric features ensure they stay connected with others like them.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>